# SQL query from table names - Continued

In [5]:
from openai import OpenAI
import os
from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv())

OPENAI_API_KEY  = os.getenv('OPENAI_API_KEY')

## The old Prompt

In [6]:
#The old prompt
old_context = [ {'role':'system', 'content':"""
you are a bot to assist in create SQL commands, all your answers should start with \
this is your SQL, and after that an SQL that can do what the user request. \
Your Database is composed by a SQL database with some tables. \
Try to maintain the SQL order simple.
Put the SQL command in white letters with a black background, and just after \
a simple and concise text explaining how it works.
If the user ask for something that can not be solved with an SQL Order \
just answer something nice and simple, maximum 10 words, asking him for something that \
can be solved with SQL.
"""} ]

old_context.append( {'role':'system', 'content':"""
first table:
{
  "tableName": "employees",
  "fields": [
    {
      "nombre": "ID_usr",
      "tipo": "int"
    },
    {
      "nombre": "name",
      "tipo": "varchar"
    }
  ]
}
"""
})

old_context.append( {'role':'system', 'content':"""
second table:
{
  "tableName": "salary",
  "fields": [
    {
      "nombre": "ID_usr",
      "type": "int"
    },
    {
      "name": "year",
      "type": "date"
    },
    {
      "name": "salary",
      "type": "float"
    }
  ]
}
"""
})

old_context.append( {'role':'system', 'content':"""
third table:
{
  "tablename": "studies",
  "fields": [
    {
      "name": "ID",
      "type": "int"
    },
    {
      "name": "ID_usr",
      "type": "int"
    },
    {
      "name": "educational_level",
      "type": "int"
    },
    {
      "name": "Institution",
      "type": "varchar"
    },
    {
      "name": "Years",
      "type": "date"
    }
    {
      "name": "Speciality",
      "type": "varchar"
    }
  ]
}
"""
})

## New Prompt.
We are going to improve it following the instructions of a Paper from the Ohaio University: [How to Prompt LLMs for Text-to-SQL: A Study in Zero-shot, Single-domain, and Cross-domain Settings](https://arxiv.org/abs/2305.11853). I recommend you read that paper.

For each table, we will define the structure using the same syntax as in a SQL create table command, and add the sample rows of the content.

Finally, at the end of the prompt, we'll include some example queries with the SQL that the model should generate. This technique is called Few-Shot Samples, in which we provide the prompt with some examples to assist it in generating the correct SQL.


In [7]:
context = [ {'role':'system', 'content':"""
You are an assistant that generates SQL queries from natural language questions.

Below are the tables in the database:

CREATE TABLE employees (
  ID_usr INTEGER,
  name TEXT
);

CREATE TABLE salary (
  ID_usr INTEGER,
  year DATE,
  salary FLOAT
);

CREATE TABLE studies (
  ID INTEGER,
  ID_usr INTEGER,
  educational_level INTEGER,
  Institution TEXT,
  Years DATE,
  Speciality TEXT
);

Sample data is available in each table.

"""} ]




In [8]:
#FEW SHOT SAMPLES
context.append({'role':'system', 'content':"""
-- Example 1:
Question: What is the name of the employee who earns the highest salary?
SQL:
SELECT e.name
FROM employees e
JOIN salary s ON e.ID_usr = s.ID_usr
ORDER BY s.salary DESC
LIMIT 1;

-- Example 2:
Question: What is the average salary per institution?
SQL:
SELECT st.Institution, AVG(sa.salary) AS avg_salary
FROM studies st
JOIN employees e ON st.ID_usr = e.ID_usr
JOIN salary sa ON e.ID_usr = sa.ID_usr
GROUP BY st.Institution
ORDER BY avg_salary DESC;

-- Example 3:
Question: List all employees and their education level.
SQL:
SELECT e.name, st.educational_level
FROM employees e
JOIN studies st ON e.ID_usr = st.ID_usr;
"""})

In [9]:
#Functio to call the model.
def return_CCRMSQL(user_message, context):
    client = OpenAI(
    # This is the default and can be omitted
    api_key=OPENAI_API_KEY,
)

    newcontext = context.copy()
    newcontext.append({'role':'user', 'content':"question: " + user_message})

    response = client.chat.completions.create(
            model="gpt-3.5-turbo",
            messages=newcontext,
            temperature=0,
        )

    return (response.choices[0].message.content)

## NL2SQL Samples
We're going to review some examples generated with the old prompt and others with the new prompt.

In [10]:
#new
context_user = context.copy()
print(return_CCRMSQL("What is the name of the employee who earns the highest salary?", context_user))

SQL:
SELECT e.name
FROM employees e
JOIN salary s ON e.ID_usr = s.ID_usr
ORDER BY s.salary DESC
LIMIT 1;


In [12]:
#old
old_context_user = old_context.copy()
print(return_CCRMSQL("What is the name of the employee who earns the highest salary?", old_context_user))

This is your SQL:
```sql
SELECT e.name
FROM employees e
JOIN salary s ON e.ID_usr = s.ID_usr
ORDER BY s.salary DESC
LIMIT 1;
```

This SQL query selects the name of the employee who earns the highest salary by joining the "employees" table with the "salary" table on the employee ID. It then orders the result by salary in descending order and limits the output to only one row, which represents the employee with the highest salary.


In [11]:
#new
print(return_CCRMSQL("What is the institution with the highest average salary?", context_user))

SQL:
SELECT st.Institution, AVG(sa.salary) AS avg_salary
FROM studies st
JOIN employees e ON st.ID_usr = e.ID_usr
JOIN salary sa ON e.ID_usr = sa.ID_usr
GROUP BY st.Institution
ORDER BY avg_salary DESC
LIMIT 1;


In [16]:
#old
print(return_CCRMSQL("Return only the institution name with the highest average salary using studies and salary tables.", old_context_user))

This is your SQL:
```sql
SELECT s.Institution
FROM studies s
JOIN salary sa ON s.ID_usr = sa.ID_usr
GROUP BY s.Institution
ORDER BY AVG(sa.salary) DESC
LIMIT 1;
```

This SQL query joins the `studies` and `salary` tables on the `ID_usr` column. It then calculates the average salary for each institution, orders the results in descending order based on the average salary, and finally returns the institution name with the highest average salary.


# Exercise
 - Complete the prompts similar to what we did in class. 
     - Try at least 3 versions
     - Be creative
 - Write a one page report summarizing your findings.
     - Were there variations that didn't work well? i.e., where GPT either hallucinated or wrong.
     - What did you learn?

## V1

In [17]:
#new
print(return_CCRMSQL("List all employees along with their salary and field of study.", context_user))

SQL:
SELECT e.name, s.salary, st.Speciality
FROM employees e
JOIN salary s ON e.ID_usr = s.ID_usr
JOIN studies st ON e.ID_usr = st.ID_usr;


In [18]:
#old
print(return_CCRMSQL("List all employees along with their salary and field of study.", old_context_user))

This is your SQL:
```sql
SELECT e.name, s.salary, st.educational_level, st.Institution, st.Speciality
FROM employees e
JOIN salary s ON e.ID_usr = s.ID_usr
JOIN studies st ON e.ID_usr = st.ID_usr;
```

This SQL query retrieves the name of employees, their salary, educational level, institution, and speciality by joining the employees table with the salary and studies tables based on the employee ID.


## V2

In [19]:
#new
print(return_CCRMSQL("How many employees studied at each institution?", context_user))

SQL:
SELECT st.Institution, COUNT(DISTINCT e.ID_usr) AS num_employees
FROM studies st
JOIN employees e ON st.ID_usr = e.ID_usr
GROUP BY st.Institution;


In [20]:
#old
print(return_CCRMSQL("How many employees studied at each institution?", old_context_user))

This is your SQL:
```sql
SELECT Institution, COUNT(ID_usr) AS num_employees
FROM studies
GROUP BY Institution;
```

This SQL query selects the institution from the studies table and counts the number of employees (ID_usr) that studied at each institution. The result will show the institution name along with the count of employees who studied there.


## V3

In [21]:
#new
print(return_CCRMSQL("Which employee had the highest salary in the year 2022?", context_user))

SQL:
SELECT e.name
FROM employees e
JOIN salary s ON e.ID_usr = s.ID_usr
WHERE s.year = '2022'
ORDER BY s.salary DESC
LIMIT 1;


In [22]:
#old
print(return_CCRMSQL("Which employee had the highest salary in the year 2022?", old_context_user))

This is your SQL:
```sql
SELECT e.name
FROM employees e
JOIN salary s ON e.ID_usr = s.ID_usr
WHERE s.year = '2022'
ORDER BY s.salary DESC
LIMIT 1;
```

This SQL query retrieves the name of the employee who had the highest salary in the year 2022 by joining the "employees" and "salary" tables on the employee ID. It then filters the results to only include salaries from the year 2022, orders them in descending order, and limits the output to the top result, which corresponds to the employee with the highest salary in 2022.


# One Page Report – Comparing Old vs. New Prompts for NL2SQL

## Experiment 1: List all employees along with their salary and field of study

- **New Prompt Output:**
  - Returned name, salary, and speciality.
- **Old Prompt Output:**
  - Included more details (educational level, institution, speciality).
- ✅ Both answers were correct. The old prompt gave extra useful fields.

---

## Experiment 2: How many employees studied at each institution?

- **New Prompt Output:**
  - Used `COUNT(DISTINCT e.ID_usr)` with a proper join.
- **Old Prompt Output:**
  - Only used the `studies` table without joins.
- ⚠️ The old prompt gave a simpler but less accurate result.
- ✅ New prompt provided a more complete answer.

---

## Experiment 3: Which employee had the highest salary in the year 2022?

- **Both Outputs:**
  - Identical SQL with correct logic and structure.
  - Used a `WHERE` clause with correct filtering and `ORDER BY`.
- ✅ No difference in quality.

---

## Findings:

- The **new prompt** gives more precise and complete SQL, especially for questions involving joins.
- The **old prompt** still works well for simple queries but may skip details or relationships.
- Including examples in the prompt (few-shot) helps guide GPT to better results.

---

## What I Learned:

- GPT handles SQL generation well with clear prompts and table definitions.
- Few-shot examples improve accuracy in complex queries.
- Even small differences in prompt structure can affect SQL quality and completeness.